# 🎯 BƯỚC 1 — Tấn Công Tái Tạo Bị Động (Passive Reconstruction Attack) trên Google Colab

### 📖 Ý nghĩa & Mục tiêu của Bước 1:
- **Mục tiêu**: Đóng băng mô hình Client $F_c$ đã huấn luyện ở Bước 0, huấn luyện Decoder $D_\phi$ giải bài toán hồi quy tái tạo ảnh từ biểu diễn trung gian (IR) $z = F_c(x)$:
  $$\min_\phi \; \mathbb{E}\big[\|D_\phi(F_c(x)) - x\|^2\big]$$
- **Tiêu chí nghiệm thu (Acceptance Criteria)**:
  - [x] Huấn luyện Decoder 30 epoch ổn định.
  - [x] **PSNR > 20 dB** (kỳ vọng 25–30 dB) và **SSIM > 0.6** (kỳ vọng 0.8+) trên tập test.
  - [x] Ảnh tái tạo $\hat{x}$ nhìn rõ đối tượng bằng mắt thường (đúng hình dạng, đúng lớp).
  - [x] Lưu checkpoint `b1_decoder.pt` thành công (sẽ tái sử dụng ở Bước 5).
- **Ý nghĩa khoa học đối với đề tài**:
  - Chứng minh tiền đề cốt lõi của đề tài: Biểu diễn trung gian (IR) nông của Vanilla Split Learning ($64 \times 32 \times 32$) **hoàn toàn bị đảo ngược được** khi không có cơ chế phòng thủ.
  - Thiết lập mức sàn cơ sở (attacker baseline) để chứng minh tính hiệu quả của cơ chế mã hóa tri giác Task-Aware ở các bước tiếp theo.

---  
## 1. Kiểm tra Môi trường & GPU (Tesla T4)

In [ ]:
!nvidia-smi

import torch
print(f"\nPyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Thiết bị GPU    : {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ CẢNH BÁO: Bạn đang chạy trên CPU! Hãy vào 'Runtime' -> 'Change runtime type' -> chọn 'T4 GPU' để train nhanh.")

---  
## 2. Kết nối Google Drive (Lưu trữ Checkpoint & Kết quả vĩnh viễn)

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
DRIVE_STEP1_DIR = '/content/drive/MyDrive/AbReTAPE_Step1'
os.makedirs(DRIVE_STEP1_DIR, exist_ok=True)
print(f"✅ Thư mục lưu kết quả Bước 1 trên Google Drive: {DRIVE_STEP1_DIR}")

---  
## 3. Thiết lập Codebase & Cài đặt Thư viện

In [ ]:
# Cài đặt scikit-image và lpips (độ đo cảm nhận)
!pip install -q scikit-image lpips matplotlib pandas torchvision

import os
# Kiểm tra thư mục dự án
if not os.path.exists('step1'):
    print("⚠️ Đang tìm kiếm mã nguồn dự án... Nếu bạn clone từ GitHub hãy dùng lệnh dưới:")
    # !git clone https://github.com/CuongBien/AbReTAPE.git /content/AbReTAPE
    # %cd /content/AbReTAPE
else:
    print("✅ Đã tìm thấy thư mục 'step1'. Codebase sẵn sàng!")

!ls -la

---  
## 4. Tải Dữ liệu CIFAR-10 & Chạy Smoke Test Bước 1

In [ ]:
# Tải dữ liệu siêu tốc bằng script đa luồng có resume
!python step0/download_cifar.py

# Chạy kiểm thử đơn vị cho Bước 1 (kiểm tra forward, backward decoder và freeze client)
!python step1/test_step1.py

---  
## 5. Huấn luyện Decoder Tái tạo Bị động (30 Epochs)
- Client $F_c$ từ Bước 0 được đóng băng hoàn toàn.
- Đánh giá định kỳ PSNR, SSIM, LPIPS sau mỗi 5 epoch.
- Checkpoint, ảnh lưới đối chứng và đồ thị được lưu trực tiếp vào Google Drive.

In [ ]:
# Tìm đường dẫn checkpoint Client từ Bước 0 (ưu tiên trong Google Drive nếu đã train ở Bước 0)
import os
client_ckpt = "/content/drive/MyDrive/AbReTAPE_Step0/best_b0_vanilla.pt"
if not os.path.exists(client_ckpt):
    client_ckpt = "/content/drive/MyDrive/AbReTAPE_Step0/b0_vanilla.pt"
if not os.path.exists(client_ckpt):
    client_ckpt = "step0/b0_vanilla.pt"

print(f"Using Client Checkpoint: {client_ckpt}")

!python step1/main.py \
    --epochs 30 \
    --batch-size 128 \
    --lr 0.001 \
    --eval-freq 5 \
    --client-ckpt {client_ckpt} \
    --data-dir step0/data \
    --output /content/drive/MyDrive/AbReTAPE_Step1/b1_decoder.pt \
    --save-best /content/drive/MyDrive/AbReTAPE_Step1/best_b1_decoder.pt \
    --checkpoint /content/drive/MyDrive/AbReTAPE_Step1/last_checkpoint_b1.pt \
    --history-file /content/drive/MyDrive/AbReTAPE_Step1/step1_history.json \
    --plot-file /content/drive/MyDrive/AbReTAPE_Step1/attack_curves.png \
    --grid-file /content/drive/MyDrive/AbReTAPE_Step1/reconstruction_grid.png

---  
## 6. Khôi phục Huấn luyện tiếp tục (Resume Training nếu cần)

In [ ]:
# Chạy cell này nếu phiên Colab trước bị ngắt để tiếp tục từ checkpoint dở dang
!python step1/main.py \
    --resume \
    --epochs 30 \
    --batch-size 128 \
    --lr 0.001 \
    --eval-freq 5 \
    --client-ckpt {client_ckpt} \
    --data-dir step0/data \
    --output /content/drive/MyDrive/AbReTAPE_Step1/b1_decoder.pt \
    --save-best /content/drive/MyDrive/AbReTAPE_Step1/best_b1_decoder.pt \
    --checkpoint /content/drive/MyDrive/AbReTAPE_Step1/last_checkpoint_b1.pt \
    --history-file /content/drive/MyDrive/AbReTAPE_Step1/step1_history.json \
    --plot-file /content/drive/MyDrive/AbReTAPE_Step1/attack_curves.png \
    --grid-file /content/drive/MyDrive/AbReTAPE_Step1/reconstruction_grid.png

---  
## 7. Đánh giá Trực quan Bằng Mắt Thường: Ảnh Gốc vs Ảnh Tái Tạo (Reconstruction Grid)

In [ ]:
from IPython.display import Image, display
import os

grid_file = "/content/drive/MyDrive/AbReTAPE_Step1/reconstruction_grid.png"
if not os.path.exists(grid_file):
    grid_file = "step1/reconstruction_grid.png"

if os.path.exists(grid_file):
    print("👁️ SO SÁNH TRỰC QUAN: ẢNH GỐC (HÀNG 1) VS ẢNH TÁI TẠO (HÀNG 2)")
    display(Image(filename=grid_file, width=950))
else:
    print(f"⚠️ Không tìm thấy file lưới ảnh tại {grid_file}")

---  
## 8. Đồ thị Quá trình Huấn luyện Tấn công (MSE, PSNR, SSIM, LPIPS)

In [ ]:
curve_file = "/content/drive/MyDrive/AbReTAPE_Step1/attack_curves.png"
if not os.path.exists(curve_file):
    curve_file = "step1/attack_curves.png"

if os.path.exists(curve_file):
    print("📈 ĐỒ THỊ CHỈ SỐ TẤN CÔNG (MSE, PSNR, SSIM, LPIPS/TIME)")
    display(Image(filename=curve_file, width=950))
else:
    print(f"⚠️ Không tìm thấy file đồ thị tại {curve_file}")

---  
## 9. Thống kê Chi tiết & Đối chiếu Tiêu chí Nghiệm thu

In [ ]:
import json
import pandas as pd
import os

hist_file = "/content/drive/MyDrive/AbReTAPE_Step1/step1_history.json"
if not os.path.exists(hist_file):
    hist_file = "step1/step1_history.json"

if os.path.exists(hist_file):
    with open(hist_file, "r", encoding="utf-8") as f:
        data = json.load(f)
    df = pd.DataFrame(data)
    
    eval_df = df[df["test_psnr"].notna()].copy()
    best_row = eval_df.loc[eval_df["test_psnr"].idxmax()]
    
    print("=" * 50)
    print("🏆 TỔNG KẾT KẾT QUẢ BƯỚC 1 (ATTACKER BASELINE)")
    print("=" * 50)
    print(f"- Best Epoch          : {int(best_row['epoch'])}")
    print(f"- Train MSE Loss      : {best_row['train_mse']:.5f}")
    print(f"- Test MSE Loss       : {best_row['test_mse']:.5f}")
    print(f"- Best PSNR           : {best_row['test_psnr']:.2f} dB (Tiêu chuẩn: > 20 dB) -> {'✅ ĐẠT' if best_row['test_psnr'] > 20.0 else '❌ CHƯA ĐẠT'}")
    print(f"- Best SSIM           : {best_row['test_ssim']:.4f} (Tiêu chuẩn: > 0.60) -> {'✅ ĐẠT' if best_row['test_ssim'] > 0.60 else '❌ CHƯA ĐẠT'}")
    if 'test_lpips' in best_row and pd.notna(best_row['test_lpips']):
        print(f"- Test LPIPS          : {best_row['test_lpips']:.4f}")
    print("=" * 50)
    
    display(eval_df)
else:
    print(f"Chưa tìm thấy file lịch sử {hist_file}")